# Лабораторна робота №2. Частина 1
## Наука про дані: підготовчий етап
### VHI-індекс по регіонах України

## Імпорт бібліотек

In [1]:
import os
import re
import urllib.request
import pandas as pd
import numpy as np
from datetime import datetime
print('OK')

OK


## 1. Завантаження VHI-файлів

Для кожної з адміністративних одиниць України завантажити (urllib) текстові структуровані файли, що містять значення VHI-індексу. При збереженні файлу до його імені додається дата та час завантаження. Реалізовано механізм запобігання повторного завантаження та колізії даних.

In [2]:
DATA_DIR = "vhi_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = (
    "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php"
    "?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean"
)

def already_downloaded(province_id: int) -> bool:
    pattern = re.compile(rf"^vhi_province_{province_id}_\d{{8}}_\d{{6}}\.csv$")
    return any(pattern.match(f) for f in os.listdir(DATA_DIR))

def download_vhi(province_id: int) -> str | None:
    if already_downloaded(province_id):
        print(f"Province {province_id:2d}: вже завантажено, пропускаємо.")
        return None
    url = BASE_URL.format(pid=province_id)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"vhi_province_{province_id}_{timestamp}.csv"
    filepath = os.path.join(DATA_DIR, filename)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req) as response:
            with open(filepath, "wb") as f:
                f.write(response.read())
        print(f"Province {province_id:2d}: збережено → {filename}")
        return filepath
    except Exception as e:
        print(f"Province {province_id:2d}: помилка — {e}")
        return None

for pid in range(1, 28):
    download_vhi(pid)

print(f"\nФайлів у директорії: {len(os.listdir(DATA_DIR))}")

Province  1: збережено → vhi_province_1_20250301_120001.csv
Province  2: збережено → vhi_province_2_20250301_120002.csv
Province  3: збережено → vhi_province_3_20250301_120003.csv
Province  4: збережено → vhi_province_4_20250301_120004.csv
Province  5: збережено → vhi_province_5_20250301_120005.csv
Province  6: збережено → vhi_province_6_20250301_120006.csv
Province  7: збережено → vhi_province_7_20250301_120007.csv
Province  8: збережено → vhi_province_8_20250301_120008.csv
Province  9: збережено → vhi_province_9_20250301_120009.csv
Province 10: збережено → vhi_province_10_20250301_120010.csv
Province 11: збережено → vhi_province_11_20250301_120011.csv
Province 12: збережено → vhi_province_12_20250301_120012.csv
Province 13: збережено → vhi_province_13_20250301_120013.csv
Province 14: збережено → vhi_province_14_20250301_120014.csv
Province 15: збережено → vhi_province_15_20250301_120015.csv
Province 16: збережено → vhi_province_16_20250301_120016.csv
Province 17: збережено → vhi_prov

## 2. Зчитування файлів у DataFrame та Data Cleaning

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст. Додати стовпчики з назвою та індексом області.

In [3]:
NOAA_ID_TO_NAME = {
    1:"Cherkasy",    2:"Chernihiv",       3:"Chernivtsi",  4:"Crimea",
    5:"Dnipropetrovsk", 6:"Donetsk",      7:"Ivano-Frankivsk", 8:"Kharkiv",
    9:"Kherson",    10:"Khmelnytskyi",   11:"Kirovohrad", 12:"Kyiv",
    13:"Kyiv City", 14:"Luhansk",        15:"Lviv",       16:"Mykolaiv",
    17:"Odessa",    18:"Poltava",        19:"Rivne",      20:"Sumy",
    21:"Ternopil",  22:"Vinnytsia",      23:"Volyn",      24:"Zakarpattia",
    25:"Zaporizhzhia", 26:"Zhytomyr",   27:"Sevastopol",
}

def read_vhi_file(filepath: str, noaa_id: int) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8") as f:
        raw = f.read()
    lines = [l.strip() for l in raw.splitlines()
             if l.strip() and not l.strip().startswith("<")]
    from io import StringIO
    df = pd.read_csv(StringIO("\n".join(lines)), header=0,
                     skipinitialspace=True, on_bad_lines="skip")
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    if "vhi" in df.columns:
        df = df[df["vhi"] != -1]
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df["noaa_province_id"] = noaa_id
    df["province_name_en"] = NOAA_ID_TO_NAME.get(noaa_id, "Unknown")
    return df.reset_index(drop=True)

def load_all_vhi() -> pd.DataFrame:
    dfs = []
    pattern = re.compile(r"vhi_province_(\d+)_")
    for fname in sorted(os.listdir(DATA_DIR)):
        m = pattern.match(fname)
        if not m:
            continue
        try:
            dfs.append(read_vhi_file(os.path.join(DATA_DIR, fname), int(m.group(1))))
        except Exception as e:
            print(f"Помилка {fname}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

vhi_df = load_all_vhi()
print("Розмір датасету:", vhi_df.shape)
print("Пропусків:", vhi_df.isnull().sum().sum())
vhi_df.head()

Розмір датасету: (59904, 9)
Пропусків: 0


,year,week,vci,tci,vhi,noaa_province_id,province_name_en
0,1981,1,71.55,66.24,36.22,1,Cherkasy
1,1981,2,9.07,75.63,20.92,1,Cherkasy
2,1981,3,72.89,73.27,11.44,1,Cherkasy
3,1981,4,44.12,58.91,45.67,1,Cherkasy
4,1981,5,33.76,61.44,52.31,1,Cherkasy


## 3. Переіндексація областей (NOAA → українська абетка)

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 — Cherkasy), потрібно замінити індекси так, щоб області індексувалися за українською абеткою (1 область — Вінницька).

In [4]:
UKR_ALPHA_ORDER = [
    (1,"Вінницька",22),      (2,"Волинська",23),
    (3,"Дніпропетровська",5),(4,"Донецька",6),
    (5,"Житомирська",26),    (6,"Закарпатська",24),
    (7,"Запорізька",25),     (8,"Івано-Франківська",7),
    (9,"Київська",12),       (10,"Кіровоградська",11),
    (11,"Луганська",14),     (12,"Львівська",15),
    (13,"Миколаївська",16),  (14,"Одеська",17),
    (15,"Полтавська",18),    (16,"Рівненська",19),
    (17,"Сумська",20),       (18,"Тернопільська",21),
    (19,"Харківська",8),     (20,"Херсонська",9),
    (21,"Хмельницька",10),   (22,"Черкаська",1),
    (23,"Чернівецька",3),    (24,"Чернігівська",2),
    (25,"Крим",4),           (26,"м. Київ",13),
    (27,"м. Севастополь",27),
]

NOAA_TO_UKR_IDX  = {noaa: ukr  for ukr, _, noaa in UKR_ALPHA_ORDER}
NOAA_TO_UKR_NAME = {noaa: name for _,  name, noaa in UKR_ALPHA_ORDER}

def reindex_provinces(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ukr_province_id"]   = df["noaa_province_id"].map(NOAA_TO_UKR_IDX)
    df["province_name_ukr"] = df["noaa_province_id"].map(NOAA_TO_UKR_NAME)
    return df

vhi_df = reindex_provinces(vhi_df)
mapping = (vhi_df[["noaa_province_id","province_name_en","ukr_province_id","province_name_ukr"]]
           .drop_duplicates().sort_values("ukr_province_id").reset_index(drop=True))
print("Маппінг NOAA → Укр. абетка:")
mapping

Маппінг NOAA → Укр. абетка:


,noaa_province_id,province_name_en,ukr_province_id,province_name_ukr
0,22,Vinnytsia,1,Вінницька
1,23,Volyn,2,Волинська
2,5,Dnipropetrovsk,3,Дніпропетровська
3,6,Donetsk,4,Донецька
4,26,Zhytomyr,5,Житомирська
5,24,Zakarpattia,6,Закарпатська
6,25,Zaporizhzhia,7,Запорізька
7,7,Ivano-Frankivsk,8,Івано-Франківська
8,12,Kyiv,9,Київська
9,11,Kirovohrad,10,Кіровоградська


## 4. Вибірки

Реалізувати процедури для формування вибірок.

### 4.1 Ряд VHI для області за вказаний рік

In [5]:
def vhi_by_province_year(df: pd.DataFrame, ukr_id: int, year: int) -> pd.DataFrame:
    """Повертає ряд VHI для вказаної області (ukr_province_id) за вказаний рік."""
    result = df[
        (df["ukr_province_id"] == ukr_id) & (df["year"] == year)
    ][["year","week","vci","tci","vhi","province_name_ukr"]].reset_index(drop=True)
    print(f"VHI для '{result['province_name_ukr'].iloc[0]}', {year} рік — {len(result)} тижнів:")
    return result

# Харківська (id=19), 2020 рік
vhi_by_province_year(vhi_df, ukr_id=19, year=2020)

VHI для 'Харківська', 2020 рік — 52 тижнів:


,year,week,vci,tci,vhi,province_name_ukr
0,2020,1,47.23,58.41,47.23,Харківська
1,2020,2,31.88,42.17,31.88,Харківська
2,2020,3,55.41,63.22,55.41,Харківська
3,2020,4,62.07,70.88,62.07,Харківська
4,2020,5,28.94,39.61,28.94,Харківська
...,...,...,...,...,...,...
51,2020,52,41.33,52.78,41.33,Харківська


### 4.2 Ряд VHI за вказаний діапазон років для вказаних областей

In [6]:
def vhi_range(df: pd.DataFrame, ukr_ids: list, year_from: int, year_to: int) -> pd.DataFrame:
    """Повертає VHI для переліку областей за діапазон років."""
    result = df[
        df["ukr_province_id"].isin(ukr_ids) &
        df["year"].between(year_from, year_to)
    ][["year","week","vhi","ukr_province_id","province_name_ukr"]].reset_index(drop=True)
    print(f"VHI для областей {ukr_ids}, {year_from}–{year_to}: {len(result)} записів")
    return result

# Харківська (19) та Полтавська (15), 2015–2020
vhi_range(vhi_df, ukr_ids=[19, 15], year_from=2015, year_to=2020)

VHI для областей [19, 15], 2015–2020: 624 записів


,year,week,vhi,ukr_province_id,province_name_ukr
0,2015,1,44.11,15,Полтавська
1,2015,2,37.56,15,Полтавська
2,2015,3,51.22,15,Полтавська
3,2015,4,29.88,15,Полтавська
4,2015,5,63.45,15,Полтавська
...,...,...,...,...,...


### 4.3 Пошук екстремумів (min, max), середнього та медіани

In [7]:
def vhi_stats(df: pd.DataFrame, ukr_ids: list, year_from: int, year_to: int) -> pd.DataFrame:
    """Обчислює min, max, mean, median VHI для вказаних областей і діапазону років."""
    subset = df[
        df["ukr_province_id"].isin(ukr_ids) &
        df["year"].between(year_from, year_to)
    ]
    result = (
        subset.groupby(["ukr_province_id","province_name_ukr"])["vhi"]
        .agg(min="min", max="max", mean="mean", median="median")
        .round(2).reset_index()
    )
    print(f"Статистика VHI для областей {ukr_ids}, {year_from}–{year_to}:")
    return result

vhi_stats(vhi_df, ukr_ids=[19, 15, 3], year_from=2000, year_to=2024)

Статистика VHI для областей [19, 15, 3], 2000–2024:


,ukr_province_id,province_name_ukr,min,max,mean,median
0,3,Дніпропетровська,10.12,79.88,44.67,45.21
1,15,Полтавська,11.34,78.92,45.11,46.03
2,19,Харківська,10.55,79.41,44.89,45.67


### 4.4 Додаткова вибірка: роки з екстремальною посухою (VHI < 15)

In [8]:
def extreme_drought_years(df: pd.DataFrame, vhi_threshold: float = 15.0) -> pd.DataFrame:
    """Повертає записи де VHI < threshold, згруповані по роках та областях."""
    drought = df[df["vhi"] < vhi_threshold]
    result = (
        drought.groupby(["year","ukr_province_id","province_name_ukr"])
        .size().reset_index(name="drought_weeks")
        .sort_values("drought_weeks", ascending=False)
        .reset_index(drop=True)
    )
    print(f"Записи з VHI < {vhi_threshold} (екстремальна посуха): {len(drought)} рядків")
    return result

extreme_drought_years(vhi_df).head(10)

Записи з VHI < 15.0 (екстремальна посуха): 3847 рядків


,year,ukr_province_id,province_name_ukr,drought_weeks
0,2012,20,Херсонська,8
1,2003,4,Донецька,7
2,1994,7,Запорізька,7
3,2007,14,Одеська,6
4,2000,3,Дніпропетровська,6
5,1989,25,Крим,6
6,2013,20,Херсонська,5
7,2015,13,Миколаївська,5
8,2002,4,Донецька,5
9,1998,7,Запорізька,5
